In [ ]:
# Statistics + report from windows an earlier `dtrack local run` already pulled.
# Nothing is pulled again: Left reads <run>/_local/left/<pair>/<bucket>/left.csv,
# Right reads <run>/_local/right/<pair>/<bucket>/right.csv, both written by that run.
from pathlib import Path
import json, os

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
CONFIG = Path(os.getenv('DTRACK_CONFIG', ROOT / 'examples/e2e/config.json'))   # the config that run used
OUT = Path(os.getenv('DTRACK_LOCAL_OUT', ROOT / '.dtrack-local/runs'))           # its --out
RUN_ID = os.getenv('DTRACK_RUN_ID', 'nb-local')                                   # its --run-id (e.g. CR_BUREAU_TRIGGER)
PAIRS = [p for p in os.getenv('DTRACK_PAIRS', '').split(',') if p] or None        # subset, or every pair in the run

# DuckDB budget: Left stats + Right stats run concurrently, so two connections share the budget.
os.environ.setdefault('DTRACK_DUCKDB_MEMORY', '16GB')
os.environ.setdefault('DTRACK_DUCKDB_WORKERS', '2')

folder = OUT / RUN_ID
local = folder / '_local'
assert (folder / '_status/row_check.closed.json').is_file(), f'{folder}: Stage 1 never closed; run `dtrack local run` once first'
{'run': str(folder), 'config': str(CONFIG), 'pairs': PAIRS or 'all'}

In [ ]:
# Inventory: every manifest window needs both CSVs. A pull writes <name>.part and renames,
# so a file that exists is complete; the row counts are only for eyeballing against the run log.
def data_rows(path):
    with path.open('rb') as handle:
        return sum(1 for _ in handle) - 1

manifests = {
    path.parent.name: json.loads(path.read_text())
    for path in (folder / 'pairs').glob('*/plan.json')
}
inventory = []
for pair, manifest in manifests.items():
    if PAIRS and pair not in PAIRS:
        continue
    for window in manifest['windows']:
        left = local / 'left' / pair / window['bucket'] / 'left.csv'
        right = local / 'right' / pair / window['bucket'] / 'right.csv'
        inventory.append({
            'pair': pair, 'bucket': window['bucket'], 'columns': len(manifest['columns']),
            'left_rows': data_rows(left) if left.is_file() else None,
            'right_rows': data_rows(right) if right.is_file() else None,
        })
missing = [row for row in inventory if row['left_rows'] is None or row['right_rows'] is None]
assert inventory, 'no manifest windows for the selected pairs'
assert not missing, f'windows without both CSVs (pull them with `dtrack local run --run-id {RUN_ID}`): {missing}'
inventory

In [ ]:
# Units the crashed run already gave up on stay failed until released; release them so they are compared now.
from dtrack_right.engine.backends.store import VolumeStore

store = VolumeStore(folder)
failures = [key for key in store.list_keys('pairs/') if '/col_check/failures/' in key]
if failures:
    print(f'releasing {len(failures)} failed unit(s)')
    store.delete_keys(failures)
pair_keys = store.list_keys('pairs/')
assert not any('/col_check/failures/' in key for key in pair_keys)
{'published': sum('/units/' in key for key in pair_keys), 'results': sum('/col_check/results/' in key for key in pair_keys)}

In [ ]:
# Same local col-check as `dtrack local run`, with reuse_pulled=True: Left returns the pulled left.csv
# instead of querying the source, Right computes from the pulled right.csv instead of the warehouse
# (warehouse=None, so a missing file raises instead of pulling). land=False skips the inspection DuckDB.
# Windows whose units are all published are skipped; Right resumes from Pair units and results.
from dtrack_local import pipeline
from dtrack_local.pipeline import WarehouseTarget, open_run

run = open_run(
    CONFIG, run_id=RUN_ID, out=OUT, pairs=PAIRS, reuse_pulled=True, land=False,
    target_factory=lambda right: WarehouseTarget(None, right, local / 'right', reuse=True),
)
stage_two = pipeline.col_check(run, pull_workers=1, stat_workers=1)
assert stage_two['dead'] == 0, f"failed units: inspect pairs/*/col_check/failures under {folder} then rerun this cell"
stage_two

In [ ]:
# HTML, XLSX, CSVs and summary.json under <run>/report/; _status/run.json becomes complete.
built = pipeline.report(run)
assert 'index.html' in built['files'] and 'report.xlsx' in built['files']
{'status': built['summary']['status'], 'findings': built['summary']['findings'], 'report': str(folder / 'report')}

In [ ]:
# Optional: copy the run (never _local/) to the stable runs root, like `dtrack local run --run-s3`.
RUN_S3 = os.getenv('DTRACK_RUN_S3')
pipeline.backup(run, RUN_S3) if RUN_S3 else 'set DTRACK_RUN_S3 to back up'